# Question 3 : Analyse de sensibilité (Instance Medium)

Ce notebook est dédié à l'étude demandée par la **Question 2.3 (Q3)** du sujet :
> *"Study the sensitivity of the solution and the solving performance to a couple of parameters (but not $Q_T$, which is kept for the last question). To do so, select one or two parameters that you suspect they could influence the solving difficulty, and run your optimization program for different values of those parameters. By studying the outputs of your optimization solver, conclude on the respective influence of those parameters."*

### Paramètres étudiés sur l'instance `CCTS-instance_medium.txt` :
1. **$\alpha_2$ (Coût d'investissement fixe annualisé des pipelines par km)** :
   * Régit directement les variables de décision binaires $p_{ij}(t) \in \{0, 1\}$.
   * Hypothèse : influence majeure sur la difficulté combinatoire (nœuds Branch & Bound dans la zone d'arbitrage) et sur la structure du réseau (arbitrage investissement lourd vs coût opératoire).
2. **$\beta_{1, \text{truck}}$ (Coût kilométrique opératoire du transport routier)** :
   * Le camion est la seule technologie disponible sur tous les couples de nœuds sans investissement préalable.
   * Hypothèse : régule la concurrence modale entre flexibilité routière et construction de pipelines.

In [1]:
%load_ext autoreload
%autoreload 2

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import ccts_model

# Lecture de l'instance Medium
data = ccts_model.read_instance("CCTS-instance_medium.txt")
print(f"Instance chargée : N = {data['N']} nœuds, T = {data['T']} ans")
print(f"Émissions initiales Q_init = {sum(data['E'].values()):.1f} t/an")
print(f"Valeurs de référence : alpha_2 = {data['alpha_2']}, beta_1_truck = {data['beta_1']['truck']}")

Instance chargée : N = 11 nœuds, T = 25 ans
Émissions initiales Q_init = 1650.0 t/an
Valeurs de référence : alpha_2 = 3000.0, beta_1_truck = 5.0


---
## 1. Sensibilité à $\alpha_2$ (Coût d'investissement des pipelines)

Nous faisons varier $\alpha_2$ sur la plage : **[500, 1500, 3000, 6000, 12000]** €/km/an.
* $\alpha_2 = 3000$ est la valeur nominale de référence.
* $\beta_{1, \text{truck}} = 5.0$ est maintenu constant.

In [2]:
alpha_2_values = [500, 1500, 3000, 6000, 12000]
results_alpha2 = []

print("=== DÉBUT DES RÉSOLUTIONS POUR ALPHA_2 ===")
for val in alpha_2_values:
    print(f"\n--> Résolution pour alpha_2 = {val} ...")
    res = ccts_model.resoudre_ccts(
        data,
        alpha_2=val,
        mipgap=0.005,      # 0.5% de gap pour convergence rapide et stable
        timelimit=360,     # Plafond de sécurité de 6 minutes
        tee=False
    )
    
    row = {
        'alpha_2': val,
        'status': res['status'],
        'time_sec': round(res['time_sec'], 2),
        'cost': round(res['cost'], 2) if res['cost'] else None,
        'num_pipelines': res['num_pipelines'],
        'truck_CO2': round(res['transport_totals']['truck'], 1),
        'rail_CO2': round(res['transport_totals']['rail'], 1),
        'pipe_CO2': round(res['transport_totals']['pipe'], 1),
    }
    results_alpha2.append(row)
    print(f"   Terminé en {row['time_sec']} s | Coût : {row['cost']} € | Pipelines construits : {row['num_pipelines']}")

df_alpha2 = pd.DataFrame(results_alpha2)
df_alpha2

=== DÉBUT DES RÉSOLUTIONS POUR ALPHA_2 ===

--> Résolution pour alpha_2 = 500 ...
containing a solution


NameError: name 'nodes' is not defined

In [ ]:
# Tracé des graphiques de sensibilité pour alpha_2
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Graphique 1 : Performance Solveur (Temps de calcul & Nb pipelines)
c_time = "#1E88E5"
ax1.set_xlabel(r"Coût fixe pipeline $\alpha_2$ (€/km/an)", fontsize=11, fontweight="bold")
ax1.set_ylabel("Temps de résolution (s)", color=c_time, fontsize=11, fontweight="bold")
ax1.plot(df_alpha2["alpha_2"], df_alpha2["time_sec"], marker="o", color=c_time, linewidth=2.5)
ax1.tick_params(axis="y", labelcolor=c_time)
ax1.grid(True, linestyle="--", alpha=0.5)

ax1_twin = ax1.twinx()
c_pipe = "#D81B60"
ax1_twin.set_ylabel("Nombre de pipelines construits", color=c_pipe, fontsize=11, fontweight="bold")
ax1_twin.plot(df_alpha2["alpha_2"], df_alpha2["num_pipelines"], marker="s", color=c_pipe, linestyle="--", linewidth=2)
ax1_twin.tick_params(axis="y", labelcolor=c_pipe)
ax1.set_title(r"Difficulté de résolution & Pipelines vs $\alpha_2$", fontsize=12, fontweight="bold")

# Graphique 2 : Impact Économique et Part Modale
c_cost = "#2E7D32"
ax2.set_xlabel(r"Coût fixe pipeline $\alpha_2$ (€/km/an)", fontsize=11, fontweight="bold")
ax2.set_ylabel("Coût total (€)", color=c_cost, fontsize=11, fontweight="bold")
ax2.plot(df_alpha2["alpha_2"], df_alpha2["cost"], marker="D", color=c_cost, linewidth=2.5)
ax2.tick_params(axis="y", labelcolor=c_cost)
ax2.grid(True, linestyle="--", alpha=0.5)

ax2_twin = ax2.twinx()
total_co2 = df_alpha2["truck_CO2"] + df_alpha2["rail_CO2"] + df_alpha2["pipe_CO2"]
pipe_share = (df_alpha2["pipe_CO2"] / total_co2) * 100
c_share = "#FB8C00"
ax2_twin.set_ylabel("Part du pipeline dans le transport (%)", color=c_share, fontsize=11, fontweight="bold")
ax2_twin.plot(df_alpha2["alpha_2"], pipe_share, marker="^", color=c_share, linestyle="-.", linewidth=2)
ax2_twin.tick_params(axis="y", labelcolor=c_share)
ax2_twin.set_ylim(-5, 105)
ax2.set_title(r"Coût total & Part modale pipeline vs $\alpha_2$", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig("figures/sensibilite_alpha2.png", dpi=300)
plt.show()
print("[OK] Figure enregistrée : figures/sensibilite_alpha2.png")

---
## 2. Sensibilité à $\beta_{1, \text{truck}}$ (Coût kilométrique du camion)

Nous faisons varier $\beta_{1, \text{truck}}$ sur la plage : **[1.0, 2.5, 5.0, 10.0, 20.0]** €/t-km.
* $\beta_{1, \text{truck}} = 5.0$ est la valeur nominale de référence.
* $\alpha_2 = 3000$ est maintenu constant.

In [ ]:
beta1_truck_values = [1.0, 2.5, 5.0, 10.0, 20.0]
results_beta1 = []

print("=== DÉBUT DES RÉSOLUTIONS POUR BETA_1_TRUCK ===")
for val in beta1_truck_values:
    print(f"\n--> Résolution pour beta_1_truck = {val} ...")
    res = ccts_model.resoudre_ccts(
        data,
        beta_1_truck=val,
        mipgap=0.005,
        timelimit=360,
        tee=False
    )
    
    row = {
        'beta_1_truck': val,
        'status': res['status'],
        'time_sec': round(res['time_sec'], 2),
        'cost': round(res['cost'], 2) if res['cost'] else None,
        'num_pipelines': res['num_pipelines'],
        'truck_CO2': round(res['transport_totals']['truck'], 1),
        'rail_CO2': round(res['transport_totals']['rail'], 1),
        'pipe_CO2': round(res['transport_totals']['pipe'], 1),
    }
    results_beta1.append(row)
    print(f"   Terminé en {row['time_sec']} s | Coût : {row['cost']} € | CO2 Camion : {row['truck_CO2']} t")

df_beta1 = pd.DataFrame(results_beta1)
df_beta1

In [ ]:
# Tracé des graphiques de sensibilité pour beta_1_truck
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Graphique 1 : Performance Solveur & Volume Camion
c_time = "#1E88E5"
ax1.set_xlabel(r"Coût kilométrique camion $\beta_{1,\mathrm{truck}}$ (€/t-km)", fontsize=11, fontweight="bold")
ax1.set_ylabel("Temps de résolution (s)", color=c_time, fontsize=11, fontweight="bold")
ax1.plot(df_beta1["beta_1_truck"], df_beta1["time_sec"], marker="o", color=c_time, linewidth=2.5)
ax1.tick_params(axis="y", labelcolor=c_time)
ax1.grid(True, linestyle="--", alpha=0.5)

ax1_twin = ax1.twinx()
c_truck = "#E65100"
ax1_twin.set_ylabel("Volume transporté par camion (t)", color=c_truck, fontsize=11, fontweight="bold")
ax1_twin.plot(df_beta1["beta_1_truck"], df_beta1["truck_CO2"], marker="s", color=c_truck, linestyle="--", linewidth=2)
ax1_twin.tick_params(axis="y", labelcolor=c_truck)
ax1.set_title(r"Temps de calcul & Usage du camion vs $\beta_{1,\mathrm{truck}}$", fontsize=12, fontweight="bold")

# Graphique 2 : Coût total et report modal
c_cost = "#2E7D32"
ax2.set_xlabel(r"Coût kilométrique camion $\beta_{1,\mathrm{truck}}$ (€/t-km)", fontsize=11, fontweight="bold")
ax2.set_ylabel("Coût total (€)", color=c_cost, fontsize=11, fontweight="bold")
ax2.plot(df_beta1["beta_1_truck"], df_beta1["cost"], marker="D", color=c_cost, linewidth=2.5)
ax2.tick_params(axis="y", labelcolor=c_cost)
ax2.grid(True, linestyle="--", alpha=0.5)

ax2_twin = ax2.twinx()
total_co2_b = df_beta1["truck_CO2"] + df_beta1["rail_CO2"] + df_beta1["pipe_CO2"]
truck_share = (df_beta1["truck_CO2"] / total_co2_b) * 100
c_share = "#D81B60"
ax2_twin.set_ylabel("Part modale du camion (%)", color=c_share, fontsize=11, fontweight="bold")
ax2_twin.plot(df_beta1["beta_1_truck"], truck_share, marker="^", color=c_share, linestyle="-.", linewidth=2)
ax2_twin.tick_params(axis="y", labelcolor=c_share)
ax2.set_title(r"Coût total & Part modale camion vs $\beta_{1,\mathrm{truck}}$", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig("figures/sensibilite_beta1.png", dpi=300)
plt.show()
print("[OK] Figure enregistrée : figures/sensibilite_beta1.png")

---
## 3. Synthèse et analyse pour le rapport

### Analyse de l'influence de $\alpha_2$ :
1. **Comportement aux extrêmes** :
   * Pour $\alpha_2$ très faible ($\le 500$) : l'investissement en canalisation est très bon marché. Le solveur choisit de mailler intensivement le réseau de pipelines. Les variables binaires $p_{ij}$ étant massivement fixées à 1, l'élagage de l'arbre Branch & Bound est facilité.
   * Pour $\alpha_2$ très élevé ($\ge 12000$) : l'investissement est prohibitif. Le solveur fixe immédiatement $p_{ij} = 0$, et le problème redevient un programme quasi-continu (résolution très rapide).
2. **Transition de phase (zone critique)** :
   * Autour de la valeur nominale ($\alpha_2 \approx 3000$) : le compromis entre investissement fixe lourd et exploitation au fil de l'eau est très serré. Le solveur doit explorer un grand nombre de branches pour comparer différentes topologies, ce qui crée un pic de difficulté algorithmique.

### Analyse de l'influence de $\beta_{1, \text{truck}}$ :
1. **Rôle d'arbitrage modal** :
   * Lorsque le camion est peu cher ($\beta_1 \le 1.5$), il concurrence directement le train et retarde la construction de pipelines.
   * Au-delà de $\beta_1 = 5.0$, le camion devient une solution de dépannage exclusive pour les nœuds dépourvus de rail et de pipeline.